<a href="https://colab.research.google.com/github/madhumitha-gv/LLM-Style-Impersonation/blob/main/notebooks/Inference_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
!git clone https://github.com/madhumitha-gv/LLM-Style-Impersonation.git
%cd LLM-Style-Impersonation
!pip install -r requirements.txt -q

Cloning into 'LLM-Style-Impersonation'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 62 (delta 27), reused 38 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 39.68 KiB | 19.84 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/LLM-Style-Impersonation


In [46]:
# Cell 1: Clean up
%cd /content
!rm -rf LLM-Style-Impersonation

/content


In [48]:
# Check token is working
from google.colab import userdata
from huggingface_hub import login, whoami

token = userdata.get('HF_TOKEN')
login(token=token)
print(whoami())

{'type': 'user', 'id': '6754d52cbba07064c2841a57', 'name': 'gvenkatamadhumitha', 'fullname': 'Madhumitha', 'isPro': False, 'avatarUrl': '/avatars/85892903a842852d48be95e43617935a.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'meta-llama/Llama-3.1-8B', 'role': 'fineGrained', 'createdAt': '2026-05-20T22:59:31.837Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6754d52cbba07064c2841a57', 'type': 'user', 'name': 'gvenkatamadhumitha'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'user.webhooks.read', 'user.webhooks.write']}]}}}}


In [49]:
from huggingface_hub import login
login()

In [50]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [51]:
import re, json

with open('data/raw/m.json', 'r') as f:
    content = f.read()

fixed = re.sub(r'\}(\s*\n\s*\{)', r'},\1', content)

try:
    data = json.loads(fixed)
    print(f"Fixed! {len(data)} entries loaded successfully.")
    with open('data/raw/m.json', 'w') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("Saved.")
except json.JSONDecodeError as e:
    print(f"Still broken: {e}")

Fixed! 45 entries loaded successfully.
Saved.


In [52]:
!python src/data_loader.py

Loaded K: 36 train | 9 test
  Style fingerprint: {'avg_words': 26.6, 'avg_sentences': 1.8, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.58, 'common_openers': ['i', 'yes,', 'it']}

Loaded M: 36 train | 9 test
  Style fingerprint: {'avg_words': 9.0, 'avg_sentences': 0.0, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.14, 'common_openers': ['i', 'yes', 'love']}

--- K ---
  Train: 36 | Test: 9
  Fingerprint: {'avg_words': 26.6, 'avg_sentences': 1.8, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.58, 'common_openers': ['i', 'yes,', 'it']}
  Sample train Q: How do you relax after a long day?
  Sample train A: After a long day, I like to relax by watching my favorite movies or shows, or by...

--- M ---
  Train: 36 | Test: 9
  Fingerprint: {'avg_words': 9.0, 'avg_sentences': 0.0, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.14, 'common_openers': ['i', 'yes', 'love']}
  Sample train Q: How do you relax after a long day?
 

In [53]:
!python src/run_experiments.py

STEP 1: Loading datasets
Loaded K: 36 train | 9 test
  Style fingerprint: {'avg_words': 26.6, 'avg_sentences': 1.8, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.58, 'common_openers': ['i', 'yes,', 'it']}

Loaded M: 36 train | 9 test
  Style fingerprint: {'avg_words': 9.0, 'avg_sentences': 0.0, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.14, 'common_openers': ['i', 'yes', 'yes,']}

STEP 2: Building embedding indexes
Loading embedding model: all-MiniLM-L6-v2
Loading weights: 100% 103/103 [00:00<00:00, 1692.52it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Building FAISS index for K (36 pairs)...
Building FAISS index for M (36 pairs)...
Ind

In [58]:
# Remove login from script
with open('src/run_constrained.py', 'r') as f:
    lines = f.readlines()

# Remove lines containing userdata or login
cleaned = [l for l in lines if 'userdata' not in l and 'from huggingface_hub import login' not in l and "login(token" not in l]

with open('src/run_constrained.py', 'w') as f:
    f.writelines(cleaned)

print("Fixed. Lines removed:")
print([l.strip() for l in lines if l not in cleaned])

Fixed. Lines removed:
['from google.colab import userdata', 'from huggingface_hub import login', "login(token=userdata.get('HF_TOKEN'))"]


In [59]:
!python src/run_constrained.py

Loading datasets...
Loaded K: 36 train | 9 test
  Style fingerprint: {'avg_words': 26.6, 'avg_sentences': 1.8, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.58, 'common_openers': ['i', 'yes,', 'it']}

Loaded M: 36 train | 9 test
  Style fingerprint: {'avg_words': 9.0, 'avg_sentences': 0.0, 'uses_hedging': True, 'uses_examples': True, 'starts_with_i': 0.14, 'common_openers': ['i', 'yes', 'yes,']}

Building indexes...
Loading embedding model: all-MiniLM-L6-v2
Loading weights: 100% 103/103 [00:00<00:00, 1677.79it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Building FAISS index for K (36 pairs)...
Building FAISS index for M (36 pairs)...
Indexes ready.

Loading

In [55]:
%cd /content/LLM-Style-Impersonation
!git pull origin main
!ls src/

/content/LLM-Style-Impersonation
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 3.57 KiB | 3.57 MiB/s, done.
From https://github.com/madhumitha-gv/LLM-Style-Impersonation
 * branch            main       -> FETCH_HEAD
   bb8bdf6..3827d0c  main       -> origin/main
Updating bb8bdf6..3827d0c
Fast-forward
 src/generator.py       | 101 +++++++++++++---------
 src/run_constrained.py | 224 +++++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 287 insertions(+), 38 deletions(-)
 create mode 100644 src/run_constrained.py
contrastive.py	evaluator.py  __pycache__
data_loader.py	generator.py  run_constrained.py
embeddings.py	prompts.py    run_experiments.py


In [60]:
import shutil
import os

# Zip results folder
shutil.make_archive('/content/results', 'zip', '/content/LLM-Style-Impersonation/results')

# Copy notebook
shutil.copy('/content/LLM-Style-Impersonation/notebooks/experiments.ipynb',
            '/content/Inference_on_colab.ipynb')

print("Ready to download:")
print("  /content/results.zip")
print("  /content/Inference_on_colab.ipynb")

Ready to download:
  /content/results.zip
  /content/Inference_on_colab.ipynb
